# KidsCareerDecoder — AI preprocessing demo

Mirrors server logic in `backend/controllers/sessionController.js` (`tallyAnswers`, `normalizeCountsToScores`, `toProfilerPayload`) and career normalization (`normalizeCareerEntry`).

**Requires:** Python 3.10+ (stdlib only — no pip installs).

In [ ]:
import json
from pprint import pprint

APTITUDE_TYPES = [
    "logical",
    "creative",
    "verbal",
    "social",
    "scientific",
    "practical",
]


def tally_answers(rows):
    """Same rules as sessionController.tallyAnswers (skipped ignored; counts aptitude_type)."""
    counts = {t: 0 for t in APTITUDE_TYPES}
    answered = 0
    for a in rows:
        if a.get("skipped"):
            continue
        apt = a.get("aptitude_type")
        if apt in counts:
            counts[apt] += 1
            answered += 1
    return counts, answered


def pick_top_aptitude(counts):
    """Tie-break: alphabetically smallest aptitude among those with max count (matches JS loop)."""
    max_v = max(counts[t] for t in APTITUDE_TYPES)
    tied = [t for t in APTITUDE_TYPES if counts[t] == max_v]
    return min(tied)


def normalize_counts_to_scores(counts, total):
    """Percentages per stripe; two decimal places (matches JS)."""
    out = {}
    for t in APTITUDE_TYPES:
        pct = (counts[t] / total) * 100 if total > 0 else 0
        out[t] = round(pct * 100) / 100
    return out


def to_profiler_payload(scores_obj):
    """Field names expected by backend/services/aiProfiler.js and LLM prompts."""
    return {
        "logical_pct": scores_obj["logical"],
        "creative_pct": scores_obj["creative"],
        "verbal_pct": scores_obj["verbal"],
        "social_pct": scores_obj["social"],
        "scientific_pct": scores_obj["scientific"],
        "practical_pct": scores_obj["practical"],
    }


def build_ai_request_payload(profiler_scores, age, country):
    """Shape sent conceptually to getAptitudeProfile (age + country + six percentages)."""
    return {**profiler_scores, "age": age, "country": country}


def normalize_career_entry(c):
    """Matches normalizeCareerEntry in sessionController.js."""
    if isinstance(c, str):
        return {"title": c, "salary": "", "pathway": "", "match_reason": ""}
    if not isinstance(c, dict):
        c = {}
    return {
        "title": str(c.get("title") or ""),
        "salary": str(c.get("salary") or ""),
        "pathway": str(c.get("pathway") or ""),
        "match_reason": str(c.get("match_reason") or ""),
    }


def normalize_careers_list(careers):
    return [normalize_career_entry(x) for x in careers] if careers else []

: 

## 1) Raw answers → counts → percentages → profiler payload

Each row mimics `quiz_answers` fields used at completion: `aptitude_type`, `skipped`.

In [ ]:
sample_rows = [
    {"aptitude_type": "creative", "skipped": False},
    {"aptitude_type": "logical", "skipped": False},
    {"aptitude_type": "creative", "skipped": False},
    {"aptitude_type": "verbal", "skipped": False},
    {"aptitude_type": "skipped_question", "skipped": True},
    {"aptitude_type": "scientific", "skipped": False},
]

counts, answered = tally_answers(sample_rows)
scores = normalize_counts_to_scores(counts, answered)
profiler = to_profiler_payload(scores)
top = pick_top_aptitude(counts)

print("answered_non_skipped:", answered)
print("counts:", counts)
print("scores (%):", scores)
print("top_aptitude:", top)
print("profiler payload:")
pprint(profiler)

## 2) Full request shape (before calling OpenAI / Claude)

Production code also resolves **country** (body → IPinfo → headers → DB) and **age** from `users`; here we set them explicitly.

In [ ]:
age = 10
country = "India"

full_payload = build_ai_request_payload(profiler, age, country)
print(json.dumps(full_payload, indent=2))

## 3) After the LLM — normalize career objects

Models may return extra keys or partial objects; we coerce to the schema stored in `metadata_json.careers`.

In [ ]:
mock_llm_careers = [
    {
        "title": " UX Designer at Product Companies ",
        "salary": "₹15-35 LPA",
        "pathway": "B.Des from NID/IIT or HCI program",
        "match_reason": "Creative + logical blend fits product design.",
        "noise": "should be dropped in DB",
    },
    "Legacy string-only title",
]

cleaned = normalize_careers_list(mock_llm_careers)
pprint(cleaned)

## 4) Optional — load export from JSON file

If you export `quiz_answers` for a session as `answers.json` (list of `{aptitude_type, skipped}`), run:

```python
# with open("answers.json") as f:
#     rows = json.load(f)
# counts, answered = tally_answers(rows)
```